# Data Preprocessing — EpiShield AI

## 1. Data Loading and Preservation

This notebook performs preprocessing operations
on separate copies of the original datasets.

Original source files must remain unchanged.
All transformations and cleaning operations
must be documented.

In [1]:
import pandas as pd
import numpy as np

from extract import extract_data

# Load original datasets
raw_data = extract_data()

# Create independent working copies
df_regional_clean = raw_data["regioni"].copy(deep=True)
df_national_clean = raw_data["nazionale"].copy(deep=True)
df_population_clean = raw_data["popolazione"].copy(deep=True)
df_notes_clean = raw_data["note"].copy(deep=True)

# Verify dataset dimensions
datasets = {
    "Regional": df_regional_clean,
    "National": df_national_clean,
    "Population": df_population_clean,
    "Notes": df_notes_clean
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

regioni: 37401 rows, 30 columns
nazionale: 1781 rows, 24 columns
popolazione: 210 rows, 12 columns
note: 1617 rows, 2 columns
Regional: (37401, 30)
National: (1781, 24)
Population: (210, 12)
Notes: (1617, 2)


### 2.1 Population Dataset — Data Type Conversion

In [2]:

# Population numeric columns

population_numeric_cols = [
    "totale_genere_maschile",
    "totale_genere_femminile",
    "totale_generale"
]

# Convert numeric columns
for col in population_numeric_cols:
    df_population_clean[col] = pd.to_numeric(
        df_population_clean[col],
        errors="raise"
    )

# Verify data types
print("Updated Data Types:")
display(df_population_clean.dtypes)

# Verify converted values
print("\nNumeric Columns Summary:")
display(df_population_clean[population_numeric_cols].describe())

# Verify dataset shape
print("\nDataset Shape:", df_population_clean.shape)


Updated Data Types:


codice_regione             object
codice_nuts_1              object
descrizione_nuts_1         object
codice_nuts_2              object
denominazione_regione      object
sigla_regione              object
latitudine_regione         object
longitudine_regione        object
range_eta                  object
totale_genere_maschile      int64
totale_genere_femminile     int64
totale_generale             int64
dtype: object


Numeric Columns Summary:


,totale_genere_maschile,totale_genere_femminile,totale_generale
count,210.000000,210.000000,2.100000e+02
mean,137314.209524,144642.800000,2.819570e+05
std,147341.456365,148625.548578,2.952415e+05
min,457.000000,1307.000000,1.764000e+03
25%,28046.500000,32562.500000,6.111625e+04
50%,83024.500000,85510.500000,1.671230e+05
75%,228337.000000,245804.750000,4.738752e+05
max,805221.000000,802504.000000,1.607725e+06



Dataset Shape: (210, 12)


In [4]:
import pandas as pd
from extract import extract_data

# Load original datasets
data = extract_data()

# Population original dataset
df_population = data["popolazione"]

# Create a separate copy for preprocessing
df_population_clean = df_population.copy()

# Numeric columns
population_numeric_cols = [
    "totale_genere_maschile",
    "totale_genere_femminile",
    "totale_generale"
]

# Convert numeric columns
for col in population_numeric_cols:
    df_population_clean[col] = pd.to_numeric(
        df_population_clean[col].astype("string").str.strip(),
        errors="coerce"
    )

print("Original Shape:", df_population.shape)
print("Cleaned Shape:", df_population_clean.shape)

regioni: 37401 rows, 30 columns
nazionale: 1781 rows, 24 columns
popolazione: 210 rows, 12 columns
note: 1617 rows, 2 columns
Original Shape: (210, 12)
Cleaned Shape: (210, 12)


In [5]:

# Validate the converted Population Dataset

validation_results = []

for col in population_numeric_cols:

    original_values = pd.to_numeric(
        df_population[col].astype("string").str.strip(),
        errors="coerce"
    )

    converted_values = df_population_clean[col]

    validation_results.append({
        "column": col,
        "missing_after_conversion": converted_values.isna().sum(),
        "values_preserved": original_values.equals(converted_values),
        "negative_values": (converted_values < 0).sum()
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("Original Shape:", df_population.shape)
print("Cleaned Shape:", df_population_clean.shape)
print(
    "Shape Preserved:",
    df_population.shape == df_population_clean.shape
)


,column,missing_after_conversion,values_preserved,negative_values
0,totale_genere_maschile,0,True,0
1,totale_genere_femminile,0,True,0
2,totale_generale,0,True,0


Original Shape: (210, 12)
Cleaned Shape: (210, 12)
Shape Preserved: True


In [6]:
# Geographic columns
geo_cols = [
    "latitudine_regione",
    "longitudine_regione"
]

geo_validation = []

for col in geo_cols:

    original_values = (
        df_population[col]
        .astype("string")
        .str.strip()
    )

    numeric_values = pd.to_numeric(
        original_values,
        errors="coerce"
    )

    invalid_mask = (
        original_values.notna()
        & original_values.ne("")
        & numeric_values.isna()
    )

    geo_validation.append({
        "column": col,
        "valid_numeric": numeric_values.notna().sum(),
        "invalid_values": invalid_mask.sum(),
        "minimum": numeric_values.min(),
        "maximum": numeric_values.max()
    })

    df_population_clean[col] = numeric_values.astype("float64")

geo_validation_df = pd.DataFrame(geo_validation)

display(geo_validation_df)

print("\nUpdated Data Types:")
display(df_population_clean[geo_cols].dtypes)

,column,valid_numeric,invalid_values,minimum,maximum
0,latitudine_regione,210,0,38.115697,46.499335
1,longitudine_regione,210,0,7.320149,16.867367



Updated Data Types:


latitudine_regione     float64
longitudine_regione    float64
dtype: object

In [7]:
# Text columns
population_text_cols = [
    "codice_regione",
    "codice_nuts_1",
    "descrizione_nuts_1",
    "codice_nuts_2",
    "denominazione_regione",
    "sigla_regione",
    "range_eta"
]

# Remove leading and trailing whitespace
for col in population_text_cols:
    df_population_clean[col] = (
        df_population_clean[col]
        .astype("string")
        .str.strip()
    )

# Check whitespace standardization
whitespace_check = []

for col in population_text_cols:
    whitespace_check.append({
        "column": col,
        "remaining_outer_whitespace": (
            df_population_clean[col]
            .ne(df_population_clean[col].str.strip())
            .sum()
        )
    })

display(pd.DataFrame(whitespace_check))

,column,remaining_outer_whitespace
0,codice_regione,0
1,codice_nuts_1,0
2,descrizione_nuts_1,0
3,codice_nuts_2,0
4,denominazione_regione,0
5,sigla_regione,0
6,range_eta,0


In [8]:
# Final Population Validation

print("Original Shape:", df_population.shape)
print("Cleaned Shape:", df_population_clean.shape)

# 1. Shape validation
print(
    "\nShape Preserved:",
    df_population.shape == df_population_clean.shape
)

# 2. Missing values
missing_summary = pd.DataFrame({
    "original_missing": df_population.isna().sum(),
    "cleaned_missing": df_population_clean.isna().sum()
})

display(missing_summary)

# 3. Numeric data types
print("\nNumeric Data Types:")
display(df_population_clean[population_numeric_cols].dtypes)

# 4. Population logical consistency
expected_total = (
    df_population_clean["totale_genere_maschile"]
    + df_population_clean["totale_genere_femminile"]
)

population_check = (
    expected_total == df_population_clean["totale_generale"]
)

print("\nPopulation Consistency:")
print("Matching:", population_check.sum())
print("Mismatching:", (~population_check).sum())

# 5. Duplicate business keys
duplicate_keys = df_population_clean.duplicated(
    subset=["codice_regione", "range_eta"]
).sum()

print("\nDuplicate Region-Age Combinations:", duplicate_keys)

Original Shape: (210, 12)
Cleaned Shape: (210, 12)

Shape Preserved: True


,original_missing,cleaned_missing
codice_regione,0,0
codice_nuts_1,0,0
descrizione_nuts_1,0,0
codice_nuts_2,0,0
denominazione_regione,0,0
sigla_regione,0,0
latitudine_regione,0,0
longitudine_regione,0,0
range_eta,0,0
totale_genere_maschile,0,0



Numeric Data Types:


totale_genere_maschile     Int64
totale_genere_femminile    Int64
totale_generale            Int64
dtype: object


Population Consistency:
Matching: 210
Mismatching: 0

Duplicate Region-Age Combinations: 0


In [9]:
import os

output_dir = "../Processed Data"
os.makedirs(output_dir, exist_ok=True)

df_population_clean.to_csv(
    os.path.join(output_dir, "population_clean.csv"),
    index=False
)

print("Population Dataset saved successfully.")

Population Dataset saved successfully.


## 2. Regional Dataset Preprocessing

### Objective
This section focuses on preparing the Regional Dataset for further analysis and integration.

The preprocessing operations will be performed on a separate copy of the original dataset to ensure that the raw source data remains unchanged.

### Preprocessing Tasks
The following operations will be performed:

1. **Data Type Conversion:** Convert date columns to appropriate datetime formats and numeric columns to suitable numeric data types.
2. **Missing Value Handling:** Preserve missing values and represent them consistently without introducing incorrect assumptions or artificial values.
3. **Text Standardization:** Remove unnecessary leading and trailing whitespace from text columns while preserving the original meaning.
4. **Identifier Preservation:** Keep regional identifiers and classification codes as strings to preserve their original structure.
5. **Data Validation:** Verify data types, missing values, data consistency, and dataset dimensions after preprocessing.

### Important Considerations
- The original dataset will not be modified.
- No missing values will be imputed automatically.
- No logical inconsistencies will be corrected automatically.
- No records or columns will be deleted without a documented reason.
- All preprocessing operations will follow the findings of the Data Quality Assessment.

### Expected Outcome
A cleaned Regional Dataset that preserves the original records and structure while using appropriate data types and standardized text values.

In [10]:
print("Regional Columns:")
for col in df_regional_clean.columns:
    print(f"{col}: {df_regional_clean[col].dtype}")

Regional Columns:
data: object
stato: object
codice_regione: object
denominazione_regione: object
lat: object
long: object
ricoverati_con_sintomi: object
terapia_intensiva: object
totale_ospedalizzati: object
isolamento_domiciliare: object
totale_positivi: object
variazione_totale_positivi: object
nuovi_positivi: object
dimessi_guariti: object
deceduti: object
casi_da_sospetto_diagnostico: object
casi_da_screening: object
totale_casi: object
tamponi: object
casi_testati: object
note: object
ingressi_terapia_intensiva: object
note_test: object
note_casi: object
totale_positivi_test_molecolare: object
totale_positivi_test_antigenico_rapido: object
tamponi_test_molecolare: object
tamponi_test_antigenico_rapido: object
codice_nuts_1: object
codice_nuts_2: object


In [11]:
# Display sample values from important columns

sample_cols = [
    "data",
    "codice_regione",
    "lat",
    "long",
    "ricoverati_con_sintomi",
    "nuovi_positivi",
    "tamponi_test_molecolare",
    "note"
]

for col in sample_cols:
    print(f"\n--- {col} ---")
    print(df_regional_clean[col].head(5).tolist())


--- data ---
['2020-02-24T18:00:00', '2020-02-24T18:00:00', '2020-02-24T18:00:00', '2020-02-24T18:00:00', '2020-02-24T18:00:00']

--- codice_regione ---
['13', '17', '18', '15', '08']

--- lat ---
['42.35122196', '40.63947052', '38.90597598', '40.83956555', '44.49436681']

--- long ---
['13.39843823', '15.80514834', '16.59440194', '14.25084984', '11.341720800000001']

--- ricoverati_con_sintomi ---
['0', '0', '0', '0', '10']

--- nuovi_positivi ---
['0', '0', '0', '0', '18']

--- tamponi_test_molecolare ---
['', '', '', '', '']

--- note ---
['', '', '', '', '']


In [12]:
regional_numeric_cols = [
    "ricoverati_con_sintomi",
    "terapia_intensiva",
    "totale_ospedalizzati",
    "isolamento_domiciliare",
    "totale_positivi",
    "variazione_totale_positivi",
    "nuovi_positivi",
    "dimessi_guariti",
    "deceduti",
    "casi_da_sospetto_diagnostico",
    "casi_da_screening",
    "totale_casi",
    "tamponi",
    "casi_testati",
    "ingressi_terapia_intensiva",
    "totale_positivi_test_molecolare",
    "totale_positivi_test_antigenico_rapido",
    "tamponi_test_molecolare",
    "tamponi_test_antigenico_rapido"
]

# Count empty strings and non-numeric values
for col in regional_numeric_cols:
    values = df_regional_clean[col].astype("string").str.strip()

    empty_count = values.eq("").sum()

    non_numeric = pd.to_numeric(
        values.replace("", pd.NA),
        errors="coerce"
    )

    invalid_count = (values.ne("") & non_numeric.isna()).sum()

    print(
        f"{col}: Empty = {empty_count}, "
        f"Invalid non-empty = {invalid_count}"
    )

ricoverati_con_sintomi: Empty = 0, Invalid non-empty = 0
terapia_intensiva: Empty = 0, Invalid non-empty = 0
totale_ospedalizzati: Empty = 0, Invalid non-empty = 0
isolamento_domiciliare: Empty = 0, Invalid non-empty = 0
totale_positivi: Empty = 0, Invalid non-empty = 0
variazione_totale_positivi: Empty = 0, Invalid non-empty = 0
nuovi_positivi: Empty = 0, Invalid non-empty = 0
dimessi_guariti: Empty = 0, Invalid non-empty = 0
deceduti: Empty = 0, Invalid non-empty = 0
casi_da_sospetto_diagnostico: Empty = 33999, Invalid non-empty = 0
casi_da_screening: Empty = 33999, Invalid non-empty = 0
totale_casi: Empty = 0, Invalid non-empty = 0
tamponi: Empty = 0, Invalid non-empty = 0
casi_testati: Empty = 1155, Invalid non-empty = 0
ingressi_terapia_intensiva: Empty = 5943, Invalid non-empty = 0
totale_positivi_test_molecolare: Empty = 6846, Invalid non-empty = 0
totale_positivi_test_antigenico_rapido: Empty = 6846, Invalid non-empty = 0
tamponi_test_molecolare: Empty = 6846, Invalid non-empty

In [13]:
# Convert numeric columns while preserving missing values

for col in regional_numeric_cols:
    values = df_regional_clean[col].astype("string").str.strip()

    values = values.replace("", pd.NA)

    df_regional_clean[col] = pd.to_numeric(
        values,
        errors="raise"
    ).astype("Int64")

print("Numeric conversion completed successfully.")

print("\nData Types:")
print(df_regional_clean[regional_numeric_cols].dtypes)


Numeric conversion completed successfully.

Data Types:
ricoverati_con_sintomi                    Int64
terapia_intensiva                         Int64
totale_ospedalizzati                      Int64
isolamento_domiciliare                    Int64
totale_positivi                           Int64
variazione_totale_positivi                Int64
nuovi_positivi                            Int64
dimessi_guariti                           Int64
deceduti                                  Int64
casi_da_sospetto_diagnostico              Int64
casi_da_screening                         Int64
totale_casi                               Int64
tamponi                                   Int64
casi_testati                              Int64
ingressi_terapia_intensiva                Int64
totale_positivi_test_molecolare           Int64
totale_positivi_test_antigenico_rapido    Int64
tamponi_test_molecolare                   Int64
tamponi_test_antigenico_rapido            Int64
dtype: object


In [14]:
print("Numeric Validation:")

print(
    "All numeric columns converted:",
    all(
        str(df_regional_clean[col].dtype) == "Int64"
        for col in regional_numeric_cols
    )
)

print("\nMissing Values After Conversion:")
print(df_regional_clean[regional_numeric_cols].isna().sum())

Numeric Validation:
All numeric columns converted: True

Missing Values After Conversion:
ricoverati_con_sintomi                        0
terapia_intensiva                             0
totale_ospedalizzati                          0
isolamento_domiciliare                        0
totale_positivi                               0
variazione_totale_positivi                    0
nuovi_positivi                                0
dimessi_guariti                               0
deceduti                                      0
casi_da_sospetto_diagnostico              33999
casi_da_screening                         33999
totale_casi                                   0
tamponi                                       0
casi_testati                               1155
ingressi_terapia_intensiva                 5943
totale_positivi_test_molecolare            6846
totale_positivi_test_antigenico_rapido     6846
tamponi_test_molecolare                    6846
tamponi_test_antigenico_rapido             684

In [15]:
# Convert geographic coordinates to float64

geo_cols = ["lat", "long"]

for col in geo_cols:
    values = df_regional_clean[col].astype("string").str.strip()
    values = values.replace("", pd.NA)

    df_regional_clean[col] = pd.to_numeric(
        values,
        errors="raise"
    ).astype("float64")

print("Geographic coordinates converted successfully.")

print("\nCoordinate Data Types:")
print(df_regional_clean[geo_cols].dtypes)

print("\nCoordinate Summary:")
print(df_regional_clean[geo_cols].describe())

Geographic coordinates converted successfully.

Coordinate Data Types:
lat     float64
long    float64
dtype: object

Coordinate Summary:
                lat          long
count  37401.000000  37401.000000
mean      43.046293     12.225955
std        2.488814      2.658983
min       38.115697      7.320149
25%       41.125596     11.121231
50%       43.616760     12.388247
75%       45.434905     13.768136
max       46.499335     16.867367


In [16]:
# Convert date column to datetime

df_regional_clean["data"] = pd.to_datetime(
    df_regional_clean["data"],
    format="%Y-%m-%dT%H:%M:%S",
    errors="raise"
)

print("Date conversion completed successfully.")

print("\nDate Data Type:")
print(df_regional_clean["data"].dtype)

print("\nDate Summary:")
print("Missing dates:", df_regional_clean["data"].isna().sum())
print("Unique dates:", df_regional_clean["data"].nunique())
print("First date:", df_regional_clean["data"].min())
print("Last date:", df_regional_clean["data"].max())

Date conversion completed successfully.

Date Data Type:
datetime64[ns]

Date Summary:
Missing dates: 0
Unique dates: 1781
First date: 2020-02-24 18:00:00
Last date: 2025-01-08 17:00:00


In [18]:
regional_text_cols = [
    "stato",
    "codice_regione",
    "denominazione_regione",
    "codice_nuts_1",
    "codice_nuts_2",
    "note",
    "note_test",
    "note_casi"
]

In [19]:
# Standardize text and identifier columns

for col in regional_text_cols:
    df_regional_clean[col] = (
        df_regional_clean[col]
        .astype("string")
        .str.strip()
    )

print("Text standardization completed successfully.")

print("\nRemaining Leading/Trailing Whitespace:")

for col in regional_text_cols:
    whitespace_count = (
        df_regional_clean[col]
        .str.contains(r"^\s|\s$", na=False)
        .sum()
    )

    print(f"{col}: {whitespace_count}")

Text standardization completed successfully.

Remaining Leading/Trailing Whitespace:
stato: 0
codice_regione: 0
denominazione_regione: 0
codice_nuts_1: 0
codice_nuts_2: 0
note: 0
note_test: 0
note_casi: 0


### Regional Dataset — Final Validation

This section verifies the results of the preprocessing operations.

The validation checks include:
- Dataset dimensions preservation.
- Expected data types.
- Missing values after conversion.
- Duplicate records.
- Date validity and chronological order.
- Coordinate validity.
- Preservation of regional identifiers.
- Logical consistency with the original Data Quality Assessment findings.

The purpose is to ensure that preprocessing has standardized the dataset without changing its original records or automatically correcting reported inconsistencies.

In [20]:
print("Original Shape:", raw_data["regioni"].shape)
print("Cleaned Shape:", df_regional_clean.shape)

print("\nData Types:")
print(df_regional_clean.dtypes)

print("\nDuplicate Rows:")
print(df_regional_clean.duplicated().sum())

Original Shape: (37401, 30)
Cleaned Shape: (37401, 30)

Data Types:
data                                      datetime64[ns]
stato                                     string[python]
codice_regione                            string[python]
denominazione_regione                     string[python]
lat                                              float64
long                                             float64
ricoverati_con_sintomi                             Int64
terapia_intensiva                                  Int64
totale_ospedalizzati                               Int64
isolamento_domiciliare                             Int64
totale_positivi                                    Int64
variazione_totale_positivi                         Int64
nuovi_positivi                                     Int64
dimessi_guariti                                    Int64
deceduti                                           Int64
casi_da_sospetto_diagnostico                       Int64
casi_da_screening   

In [21]:
print("Missing Dates:", df_regional_clean["data"].isna().sum())

print("Date Order:",
      df_regional_clean["data"].is_monotonic_increasing)

print("\nCoordinate Missing Values:")
print(df_regional_clean[["lat", "long"]].isna().sum())

print("\nLatitude Range:")
print(df_regional_clean["lat"].min(),
      df_regional_clean["lat"].max())

print("\nLongitude Range:")
print(df_regional_clean["long"].min(),
      df_regional_clean["long"].max())

Missing Dates: 0
Date Order: True

Coordinate Missing Values:
lat     0
long    0
dtype: int64

Latitude Range:
38.11569725 46.49933453

Longitude Range:
7.320149366 16.86736689


In [22]:
# Compare original and cleaned values

original_regional = raw_data["regioni"]

comparison_cols = [
    col for col in df_regional_clean.columns
    if col not in ["data", "lat", "long"] + regional_numeric_cols
]

print("Identifier and Text Preservation:")

for col in comparison_cols:
    original = (
        original_regional[col]
        .astype("string")
        .str.strip()
    )

    cleaned = df_regional_clean[col]

    print(f"{col}: {(original == cleaned).all()}")

Identifier and Text Preservation:
stato: True
codice_regione: True
denominazione_regione: True
note: True
note_test: True
note_casi: True
codice_nuts_1: True
codice_nuts_2: True


In [23]:
print("Numeric Value Preservation:")

for col in regional_numeric_cols:
    original = pd.to_numeric(
        original_regional[col].astype("string").str.strip().replace("", pd.NA),
        errors="raise"
    ).astype("Int64")

    cleaned = df_regional_clean[col]

    print(f"{col}: {original.equals(cleaned)}")

Numeric Value Preservation:
ricoverati_con_sintomi: True
terapia_intensiva: True
totale_ospedalizzati: True
isolamento_domiciliare: True
totale_positivi: True
variazione_totale_positivi: True
nuovi_positivi: True
dimessi_guariti: True
deceduti: True
casi_da_sospetto_diagnostico: True
casi_da_screening: True
totale_casi: True
tamponi: True
casi_testati: True
ingressi_terapia_intensiva: True
totale_positivi_test_molecolare: True
totale_positivi_test_antigenico_rapido: True
tamponi_test_molecolare: True
tamponi_test_antigenico_rapido: True


In [24]:
import os

output_dir = "../Processed Data"
os.makedirs(output_dir, exist_ok=True)

regional_output_path = os.path.join(
    output_dir,
    "regional_clean.csv"
)

df_regional_clean.to_csv(
    regional_output_path,
    index=False
)

print("Regional Dataset saved successfully!")
print("Saved at:", regional_output_path)

Regional Dataset saved successfully!
Saved at: ../Processed Data\regional_clean.csv


In [25]:
df_regional_check = pd.read_csv(
    regional_output_path,
    dtype=str,
    keep_default_na=False
)

print("Saved Shape:", df_regional_check.shape)
print("Expected Shape:", df_regional_clean.shape)

print(
    "Shape Preserved:",
    df_regional_check.shape == df_regional_clean.shape
)

Saved Shape: (37401, 30)
Expected Shape: (37401, 30)
Shape Preserved: True


## 3. National Dataset Preprocessing

### Objective
This section focuses on preparing the National Dataset for subsequent analysis while preserving the original source data.

### Preprocessing Tasks
1. Convert the date column into an appropriate datetime format.
2. Convert numeric columns into suitable numeric data types while preserving missing values.
3. Preserve identifiers as strings.
4. Standardize text columns by removing unnecessary leading and trailing whitespace.
5. Validate data types, missing values, dataset dimensions, and value preservation.

### Important Considerations
- Preserve all original records.
- Do not impute missing values automatically.
- Do not automatically correct logical inconsistencies.
- Preserve the original source data.
- Follow the findings documented in the Data Quality Assessment.

### Expected Outcome
A cleaned National Dataset with appropriate data types and standardized text values, ready for subsequent processing and analysis.

In [26]:
df_national_clean = raw_data["nazionale"].copy(deep=True)

print("National Shape:", df_national_clean.shape)

print("\nNational Columns:")
for col in df_national_clean.columns:
    print(f"{col}: {df_national_clean[col].dtype}")

National Shape: (1781, 24)

National Columns:
data: object
stato: object
ricoverati_con_sintomi: object
terapia_intensiva: object
totale_ospedalizzati: object
isolamento_domiciliare: object
totale_positivi: object
variazione_totale_positivi: object
nuovi_positivi: object
dimessi_guariti: object
deceduti: object
casi_da_sospetto_diagnostico: object
casi_da_screening: object
totale_casi: object
tamponi: object
casi_testati: object
note: object
ingressi_terapia_intensiva: object
note_test: object
note_casi: object
totale_positivi_test_molecolare: object
totale_positivi_test_antigenico_rapido: object
tamponi_test_molecolare: object
tamponi_test_antigenico_rapido: object


In [29]:
national_numeric_cols = [
    "ricoverati_con_sintomi",
    "terapia_intensiva",
    "totale_ospedalizzati",
    "isolamento_domiciliare",
    "totale_positivi",
    "variazione_totale_positivi",
    "nuovi_positivi",
    "dimessi_guariti",
    "deceduti",
    "casi_da_sospetto_diagnostico",
    "casi_da_screening",
    "totale_casi",
    "tamponi",
    "casi_testati",
    "ingressi_terapia_intensiva",
    "totale_positivi_test_molecolare",
    "totale_positivi_test_antigenico_rapido",
    "tamponi_test_molecolare",
    "tamponi_test_antigenico_rapido"
]

In [30]:
print("National Sample Values:")

sample_cols = [
    "data",
    "stato",
    "ricoverati_con_sintomi",
    "casi_da_sospetto_diagnostico",
    "casi_testati",
    "tamponi_test_molecolare",
    "note"
]

for col in sample_cols:
    print(f"\n--- {col} ---")
    print(df_national_clean[col].head(5).tolist())

National Sample Values:

--- data ---
['2020-02-24T18:00:00', '2020-02-25T18:00:00', '2020-02-26T18:00:00', '2020-02-27T18:00:00', '2020-02-28T18:00:00']

--- stato ---
['ITA', 'ITA', 'ITA', 'ITA', 'ITA']

--- ricoverati_con_sintomi ---
['101', '114', '128', '248', '345']

--- casi_da_sospetto_diagnostico ---
['', '', '', '', '']

--- casi_testati ---
['', '', '', '', '']

--- tamponi_test_molecolare ---
['', '', '', '', '']

--- note ---
['', '', '', '', '']


In [31]:
print("National Numeric Columns Assessment:")

for col in national_numeric_cols:
    values = df_national_clean[col].astype("string").str.strip()

    empty_count = values.eq("").sum()

    numeric_values = pd.to_numeric(
        values.replace("", pd.NA),
        errors="coerce"
    )

    invalid_count = (
        values.ne("") & numeric_values.isna()
    ).sum()

    print(
        f"{col}: Empty = {empty_count}, "
        f"Invalid non-empty = {invalid_count}"
    )

National Numeric Columns Assessment:
ricoverati_con_sintomi: Empty = 0, Invalid non-empty = 0
terapia_intensiva: Empty = 0, Invalid non-empty = 0
totale_ospedalizzati: Empty = 0, Invalid non-empty = 0
isolamento_domiciliare: Empty = 0, Invalid non-empty = 0
totale_positivi: Empty = 0, Invalid non-empty = 0
variazione_totale_positivi: Empty = 0, Invalid non-empty = 0
nuovi_positivi: Empty = 0, Invalid non-empty = 0
dimessi_guariti: Empty = 0, Invalid non-empty = 0
deceduti: Empty = 0, Invalid non-empty = 0
casi_da_sospetto_diagnostico: Empty = 1619, Invalid non-empty = 0
casi_da_screening: Empty = 1619, Invalid non-empty = 0
totale_casi: Empty = 0, Invalid non-empty = 0
tamponi: Empty = 0, Invalid non-empty = 0
casi_testati: Empty = 55, Invalid non-empty = 0
ingressi_terapia_intensiva: Empty = 283, Invalid non-empty = 0
totale_positivi_test_molecolare: Empty = 326, Invalid non-empty = 0
totale_positivi_test_antigenico_rapido: Empty = 326, Invalid non-empty = 0
tamponi_test_molecolare: E

In [32]:
# Investigate missing values in selected columns

check_cols = [
    "casi_testati",
    "ingressi_terapia_intensiva"
]

for col in check_cols:
    values = df_national_clean[col].astype("string")

    print(f"\n--- {col} ---")
    print("Empty strings:", values.eq("").sum())
    print("Whitespace-only:", values.str.strip().eq("").sum())
    print("Pandas NA:", values.isna().sum())
    print("First non-empty values:")
    print(values[values.str.strip().ne("")].head(10).tolist())


--- casi_testati ---
Empty strings: 55
Whitespace-only: 55
Pandas NA: 0
First non-empty values:
['935310', '943151', '971246', '1015494', '1052577', '1147850', '1186526', '1210639', '1237317', '1274871']

--- ingressi_terapia_intensiva ---
Empty strings: 283
Whitespace-only: 283
Pandas NA: 0
First non-empty values:
['217', '201', '192', '150', '144', '192', '152', '251', '208', '195']


In [33]:
# Convert National numeric columns

for col in national_numeric_cols:
    values = df_national_clean[col].astype("string").str.strip()

    values = values.replace("", pd.NA)

    df_national_clean[col] = pd.to_numeric(
        values,
        errors="raise"
    ).astype("Int64")

print("National numeric conversion completed successfully.")

print("\nNumeric Data Types:")
print(df_national_clean[national_numeric_cols].dtypes)

National numeric conversion completed successfully.

Numeric Data Types:
ricoverati_con_sintomi                    Int64
terapia_intensiva                         Int64
totale_ospedalizzati                      Int64
isolamento_domiciliare                    Int64
totale_positivi                           Int64
variazione_totale_positivi                Int64
nuovi_positivi                            Int64
dimessi_guariti                           Int64
deceduti                                  Int64
casi_da_sospetto_diagnostico              Int64
casi_da_screening                         Int64
totale_casi                               Int64
tamponi                                   Int64
casi_testati                              Int64
ingressi_terapia_intensiva                Int64
totale_positivi_test_molecolare           Int64
totale_positivi_test_antigenico_rapido    Int64
tamponi_test_molecolare                   Int64
tamponi_test_antigenico_rapido            Int64
dtype: object


In [34]:
print("Missing Values After Conversion:")

print(
    df_national_clean[national_numeric_cols]
    .isna()
    .sum()
)

Missing Values After Conversion:
ricoverati_con_sintomi                       0
terapia_intensiva                            0
totale_ospedalizzati                         0
isolamento_domiciliare                       0
totale_positivi                              0
variazione_totale_positivi                   0
nuovi_positivi                               0
dimessi_guariti                              0
deceduti                                     0
casi_da_sospetto_diagnostico              1619
casi_da_screening                         1619
totale_casi                                  0
tamponi                                      0
casi_testati                                55
ingressi_terapia_intensiva                 283
totale_positivi_test_molecolare            326
totale_positivi_test_antigenico_rapido     326
tamponi_test_molecolare                    326
tamponi_test_antigenico_rapido             326
dtype: int64


In [35]:
# Convert National date column

df_national_clean["data"] = pd.to_datetime(
    df_national_clean["data"],
    format="%Y-%m-%dT%H:%M:%S",
    errors="raise"
)

print("Date conversion completed successfully.")

print("\nDate Data Type:")
print(df_national_clean["data"].dtype)

print("\nDate Summary:")
print("Missing dates:", df_national_clean["data"].isna().sum())
print("Unique dates:", df_national_clean["data"].nunique())
print("First date:", df_national_clean["data"].min())
print("Last date:", df_national_clean["data"].max())

print(
    "Chronological order:",
    df_national_clean["data"].is_monotonic_increasing
)

Date conversion completed successfully.

Date Data Type:
datetime64[ns]

Date Summary:
Missing dates: 0
Unique dates: 1781
First date: 2020-02-24 18:00:00
Last date: 2025-01-08 17:00:00
Chronological order: True


In [36]:
# Standardize National text and identifier columns

national_text_cols = [
    "stato",
    "note",
    "note_test",
    "note_casi"
]

for col in national_text_cols:
    df_national_clean[col] = (
        df_national_clean[col]
        .astype("string")
        .str.strip()
    )

print("Text standardization completed successfully.")

print("\nRemaining Leading/Trailing Whitespace:")

for col in national_text_cols:
    whitespace_count = (
        df_national_clean[col]
        .str.contains(r"^\s|\s$", na=False)
        .sum()
    )

    print(f"{col}: {whitespace_count}")

Text standardization completed successfully.

Remaining Leading/Trailing Whitespace:
stato: 0
note: 0
note_test: 0
note_casi: 0


In [37]:
original_national = raw_data["nazionale"]

print("Original Shape:", original_national.shape)
print("Cleaned Shape:", df_national_clean.shape)

print("\nData Types:")
print(df_national_clean.dtypes)

print("\nDuplicate Rows:")
print(df_national_clean.duplicated().sum())

Original Shape: (1781, 24)
Cleaned Shape: (1781, 24)

Data Types:
data                                      datetime64[ns]
stato                                     string[python]
ricoverati_con_sintomi                             Int64
terapia_intensiva                                  Int64
totale_ospedalizzati                               Int64
isolamento_domiciliare                             Int64
totale_positivi                                    Int64
variazione_totale_positivi                         Int64
nuovi_positivi                                     Int64
dimessi_guariti                                    Int64
deceduti                                           Int64
casi_da_sospetto_diagnostico                       Int64
casi_da_screening                                  Int64
totale_casi                                        Int64
tamponi                                            Int64
casi_testati                                       Int64
note                  

In [38]:
print("Missing Dates:", df_national_clean["data"].isna().sum())
print("Unique Dates:", df_national_clean["data"].nunique())

print("First Date:", df_national_clean["data"].min())
print("Last Date:", df_national_clean["data"].max())

print(
    "Chronological Order:",
    df_national_clean["data"].is_monotonic_increasing
)

Missing Dates: 0
Unique Dates: 1781
First Date: 2020-02-24 18:00:00
Last Date: 2025-01-08 17:00:00
Chronological Order: True


In [39]:
print("Numeric Value Preservation:")

for col in national_numeric_cols:
    original = pd.to_numeric(
        original_national[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA),
        errors="raise"
    ).astype("Int64")

    cleaned = df_national_clean[col]

    print(f"{col}: {original.equals(cleaned)}")

Numeric Value Preservation:
ricoverati_con_sintomi: True
terapia_intensiva: True
totale_ospedalizzati: True
isolamento_domiciliare: True
totale_positivi: True
variazione_totale_positivi: True
nuovi_positivi: True
dimessi_guariti: True
deceduti: True
casi_da_sospetto_diagnostico: True
casi_da_screening: True
totale_casi: True
tamponi: True
casi_testati: True
ingressi_terapia_intensiva: True
totale_positivi_test_molecolare: True
totale_positivi_test_antigenico_rapido: True
tamponi_test_molecolare: True
tamponi_test_antigenico_rapido: True


In [40]:
national_text_cols = [
    "stato",
    "note",
    "note_test",
    "note_casi"
]

print("Text and Identifier Preservation:")

for col in national_text_cols:
    original = (
        original_national[col]
        .astype("string")
        .str.strip()
    )

    cleaned = df_national_clean[col]

    print(f"{col}: {(original == cleaned).all()}")

Text and Identifier Preservation:
stato: True
note: True
note_test: True
note_casi: True


In [41]:
import os

output_dir = "../Processed Data"
os.makedirs(output_dir, exist_ok=True)

national_output_path = os.path.join(
    output_dir,
    "national_clean.csv"
)

df_national_clean.to_csv(
    national_output_path,
    index=False
)

print("National Dataset saved successfully!")
print("Saved at:", national_output_path)

National Dataset saved successfully!
Saved at: ../Processed Data\national_clean.csv


In [42]:
df_national_check = pd.read_csv(
    national_output_path,
    dtype=str,
    keep_default_na=False
)

print("Saved Shape:", df_national_check.shape)
print("Expected Shape:", df_national_clean.shape)

print(
    "Shape Preserved:",
    df_national_check.shape == df_national_clean.shape
)

Saved Shape: (1781, 24)
Expected Shape: (1781, 24)
Shape Preserved: True


## 4. Notes Dataset Preprocessing

### Objective
This section focuses on preparing the Notes Dataset while preserving the original Italian text and its contextual information.

### Preprocessing Tasks
1. Convert the date column into the appropriate datetime format.
2. Standardize the note text by removing unnecessary leading and trailing whitespace.
3. Identify and preserve missing or whitespace-only notes.
4. Validate date uniqueness and chronological order.
5. Verify that the original text content is preserved.

### Important Considerations
- Do not translate or modify the Italian text.
- Do not remove repeated notes.
- Do not delete records with missing notes.
- Do not merge Notes with Regional based on date alone.
- Preserve all original records.

### Expected Outcome
A cleaned Notes Dataset with standardized dates and text, while preserving its original content and structure.

In [43]:
df_notes_clean = raw_data["note"].copy(deep=True)

print("Notes Shape:", df_notes_clean.shape)

print("\nColumns:")
for col in df_notes_clean.columns:
    print(f"{col}: {df_notes_clean[col].dtype}")

print("\nSample Values:")
print(df_notes_clean.head(5).to_string())

Notes Shape: (1617, 2)

Columns:
data: object
note: object

Sample Values:
                  data                                                                                                                                                 note
0  2020-03-07T17:00:00                                                                             La Provincia di Brescia ha comunicato dati parziali di 300 casi positivi
1  2020-03-10T17:00:00                                                                                                     La Regione Lombardia ha comunicato dati parziali
2  2020-06-01T17:00:00                                                                                                                  [altre date in fase di caricamento]
3  2020-06-25T17:00:00  La Regione Marche ha effettuato un ricalcolo sui decessi: n.3 decessi precedentemente segnalati non risultano classificabili come COVID-19 positivi
4  2020-06-26T17:00:00      La Regione Emilia Romagna ha effettua

In [44]:
# Inspect missing and whitespace-only values

for col in df_notes_clean.columns:
    values = df_notes_clean[col].astype("string")

    empty_count = values.eq("").sum()
    whitespace_count = values.str.strip().eq("").sum()
    pandas_na_count = values.isna().sum()

    print(f"\n--- {col} ---")
    print("Empty strings:", empty_count)
    print("Whitespace-only:", whitespace_count)
    print("Pandas NA:", pandas_na_count)


--- data ---
Empty strings: 0
Whitespace-only: 0
Pandas NA: 0

--- note ---
Empty strings: 14
Whitespace-only: 14
Pandas NA: 0


In [45]:
# Check date values before conversion

print("Date Samples:")
print(df_notes_clean["data"].head(5).tolist())

print("\nInvalid Date Count:")

parsed_dates = pd.to_datetime(
    df_notes_clean["data"],
    format="%Y-%m-%dT%H:%M:%S",
    errors="coerce"
)

print(parsed_dates.isna().sum())

Date Samples:
['2020-03-07T17:00:00', '2020-03-10T17:00:00', '2020-06-01T17:00:00', '2020-06-25T17:00:00', '2020-06-26T17:00:00']

Invalid Date Count:
0


In [46]:
# Convert Notes date column

df_notes_clean["data"] = pd.to_datetime(
    df_notes_clean["data"],
    format="%Y-%m-%dT%H:%M:%S",
    errors="raise"
)

print("Date conversion completed successfully.")

print("\nDate Data Type:")
print(df_notes_clean["data"].dtype)

print("\nDate Summary:")
print("Missing dates:", df_notes_clean["data"].isna().sum())
print("Unique dates:", df_notes_clean["data"].nunique())
print("First date:", df_notes_clean["data"].min())
print("Last date:", df_notes_clean["data"].max())

print(
    "Chronological order:",
    df_notes_clean["data"].is_monotonic_increasing
)

Date conversion completed successfully.

Date Data Type:
datetime64[ns]

Date Summary:
Missing dates: 0
Unique dates: 1617
First date: 2020-03-07 17:00:00
Last date: 2025-01-08 17:00:00
Chronological order: True


In [47]:
# Standardize Notes text

df_notes_clean["note"] = (
    df_notes_clean["note"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

print("Text standardization completed successfully.")

print("\nMissing Notes After Standardization:")
print(df_notes_clean["note"].isna().sum())

print("\nRemaining Leading/Trailing Whitespace:")
print(
    df_notes_clean["note"]
    .str.contains(r"^\s|\s$", na=False)
    .sum()
)

Text standardization completed successfully.

Missing Notes After Standardization:
14

Remaining Leading/Trailing Whitespace:
0


In [48]:
original_notes = raw_data["note"]

print("Original Shape:", original_notes.shape)
print("Cleaned Shape:", df_notes_clean.shape)

print("\nDuplicate Rows:")
print(df_notes_clean.duplicated().sum())

print("\nDate Preservation:")
original_dates = pd.to_datetime(
    original_notes["data"],
    format="%Y-%m-%dT%H:%M:%S"
)

print(
    "All dates preserved:",
    original_dates.equals(df_notes_clean["data"])
)

print("\nNote Text Preservation:")

original_text = (
    original_notes["note"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

print(
    "All notes preserved:",
    original_text.equals(df_notes_clean["note"])
)

print("\nMissing Notes:")
print(df_notes_clean["note"].isna().sum())

Original Shape: (1617, 2)
Cleaned Shape: (1617, 2)

Duplicate Rows:
0

Date Preservation:
All dates preserved: True

Note Text Preservation:
All notes preserved: True

Missing Notes:
14


In [49]:
import os

output_dir = "../Processed Data"
os.makedirs(output_dir, exist_ok=True)

notes_output_path = os.path.join(
    output_dir,
    "notes_clean.csv"
)

df_notes_clean.to_csv(
    notes_output_path,
    index=False
)

print("Notes Dataset saved successfully!")
print("Saved at:", notes_output_path)

Notes Dataset saved successfully!
Saved at: ../Processed Data\notes_clean.csv


In [50]:
df_notes_check = pd.read_csv(
    notes_output_path,
    dtype=str,
    keep_default_na=False
)

print("Saved Shape:", df_notes_check.shape)
print("Expected Shape:", df_notes_clean.shape)

print(
    "Shape Preserved:",
    df_notes_check.shape == df_notes_clean.shape
)

Saved Shape: (1617, 2)
Expected Shape: (1617, 2)
Shape Preserved: True
